In [1]:
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt
from datetime import date
import re
from tqdm import tqdm
import time
from pybedtools import BedTool
import random
import math
import pickle
import seaborn as sns
from scipy.stats import percentileofscore, pearsonr
from scipy.spatial.distance import jaccard, euclidean, cosine
import functools
import operator
import gzip
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

today = date.today()

In [ ]:
gene_hg19_bed = pd.read_pickle('../data/hg19_gene_bed.pkl')
bin_cols = ['seqname', 'start', 'end', '?']
bins_25kb = pd.read_csv('../data/Altius_100k_epilogos_hg19_WM20210518.bed', delimiter='\t', names=bin_cols)

# Use Specific Gene's closest regions to their start and end
gene = 'GATA1'

gene_chr = gene_hg19_bed.loc[gene].chr
gene_start = gene_hg19_bed.loc[gene].start
gene_end = gene_hg19_bed.loc[gene].end

print(gene_hg19_bed.loc[gene])
print(gene_start)
print(gene_end)

# region = bins_25kb.loc[(bins_25kb['seqname'] == gene_chr) & (bins_25kb['start'] <= gene_start)].iloc[-1]
region = bins_25kb.loc[(bins_25kb['seqname'] == gene_chr) & (bins_25kb['end'] >= gene_end)].iloc[0]

region

In [2]:
# Files created in region_recipe.ipynb
# motif_cosine = pd.read_pickle('../data/motif_cosine_25kb.pkl')
# print(motif_cosine.shape)
# dhs_cosine = pd.read_pickle('../data/dhs_cosine_25kb.pkl')
# print(dhs_cosine.shape)
epilogos_cosine = pd.read_pickle('../data/epilogos_cosine_25kb.pkl')
print(epilogos_cosine.shape)
print("Individual scores loaded.")

(99996, 99996)
Individual scores loaded.


In [ ]:
combined_score = pd.read_pickle('../output/combined_score.pkl')

In [4]:
print(combined_score.iloc[96548, :].idxmax())
print(combined_score.iloc[96548, :].max())

print(combined_score.describe())

NameError: name 'combined_score' is not defined

In [8]:
print(motif_cosine.iloc[96548, 96549:].idxmax())
print(motif_cosine.iloc[96548, 96549:].max())

print(dhs_cosine.iloc[96548, 96549:].idxmax())
print(dhs_cosine.iloc[96548, 96549:].max())

print(epilogos_cosine.iloc[96548, 96549:].idxmax())
print(epilogos_cosine.iloc[96548, 96549:].max())

97268
0.9295350176602266
96589
0.9946838726933157
96723
0.9837035051796214


In [ ]:
print(np.amax(motif_cosine))
# print(motif_cosine.min())
# print(motif_cosine.mean())
# print(motif_cosine.median())

In [29]:
motif_cosine.values[[np.arange(motif_cosine.shape[0])]*2] = -1
motif_cosine

/tmp/ipykernel_8224/2718107367.py:1: FutureWarning: Using a non-tuple sequence for multidimensional indexing is deprecated; use `arr[tuple(seq)]` instead of `arr[seq]`. In the future this will be interpreted as an array index, `arr[np.array(seq)]`, which will result either in an error or a different result.
  motif_cosine.values[[np.arange(motif_cosine.shape[0])]*2] = -1


,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
0,-1.000000,0.517479,0.431465,0.552631,0.757408,0.712969,0.791547,0.936062,0.0,0.298166,...,0.572527,0.537958,0.547472,0.633495,0.711792,0.575455,0.792505,0.499967,0.589655,0.567611
1,0.517479,-1.000000,0.762757,0.763772,0.696884,0.623399,0.718753,0.594868,0.0,0.490550,...,0.750978,0.755607,0.753423,0.757235,0.661968,0.759152,0.678938,0.781730,0.767918,0.760462
2,0.431465,0.762757,-1.000000,0.797089,0.711510,0.628755,0.604133,0.502751,0.0,0.549661,...,0.829889,0.827435,0.788805,0.805966,0.615844,0.737649,0.631472,0.795827,0.801242,0.835300
3,0.552631,0.763772,0.797089,-1.000000,0.769799,0.747697,0.737495,0.618745,0.0,0.505982,...,0.842288,0.857433,0.818026,0.866355,0.679252,0.795590,0.679215,0.780998,0.839428,0.834777
4,0.757408,0.696884,0.711510,0.769799,-1.000000,0.828851,0.725791,0.831603,0.0,0.479614,...,0.760627,0.746750,0.755000,0.766650,0.751872,0.710991,0.856740,0.721443,0.763542,0.765904
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,0.575455,0.759152,0.737649,0.795590,0.710991,0.684877,0.722019,0.639140,0.0,0.495989,...,0.800087,0.798722,0.854119,0.816975,0.827834,-1.000000,0.736024,0.759348,0.835130,0.823254
99992,0.792505,0.678938,0.631472,0.679215,0.856740,0.752201,0.709921,0.859421,0.0,0.463185,...,0.691116,0.677775,0.713453,0.696293,0.825627,0.736024,-1.000000,0.657401,0.710127,0.699917
99993,0.499967,0.781730,0.795827,0.780998,0.721443,0.681090,0.667499,0.570503,0.0,0.597014,...,0.818207,0.799005,0.850463,0.809266,0.633600,0.759348,0.657401,-1.000000,0.837125,0.840804
99994,0.589655,0.767918,0.801242,0.839428,0.763542,0.757376,0.756817,0.653991,0.0,0.544632,...,0.865019,0.855989,0.848752,0.880025,0.678273,0.835130,0.710127,0.837125,-1.000000,0.888681


In [16]:
dhs_cosine.values[[np.arange(dhs_cosine.shape[0])]*2] = -1

/tmp/ipykernel_8224/2157620774.py:1: FutureWarning: Using a non-tuple sequence for multidimensional indexing is deprecated; use `arr[tuple(seq)]` instead of `arr[seq]`. In the future this will be interpreted as an array index, `arr[np.array(seq)]`, which will result either in an error or a different result.
  dhs_cosine.values[[np.arange(dhs_cosine.shape[0])]*2] = -1


In [17]:
dhs_cosine

,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
0,-1.000000,0.761358,0.748400,0.761684,0.743562,0.659658,0.858965,0.772846,0.0,0.0,...,0.872731,0.829983,0.816108,0.746308,0.867297,0.789715,0.785205,0.815312,0.765065,0.754574
1,0.761358,-1.000000,0.968732,0.975338,0.913593,0.804773,0.937943,0.959300,0.0,0.0,...,0.949259,0.942148,0.948903,0.925761,0.953651,0.935597,0.951885,0.945711,0.948236,0.937352
2,0.748400,0.968732,-1.000000,0.986215,0.957561,0.788415,0.925018,0.941836,0.0,0.0,...,0.929346,0.920859,0.935949,0.911516,0.954178,0.927239,0.953379,0.941838,0.947014,0.923896
3,0.761684,0.975338,0.986215,-1.000000,0.967456,0.801399,0.952267,0.959546,0.0,0.0,...,0.948829,0.947158,0.963638,0.940397,0.968123,0.949468,0.968734,0.961725,0.961719,0.948031
4,0.743562,0.913593,0.957561,0.967456,-1.000000,0.782432,0.947166,0.933370,0.0,0.0,...,0.922709,0.925652,0.948014,0.906046,0.947865,0.931854,0.951266,0.950049,0.939495,0.920155
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,0.789715,0.935597,0.927239,0.949468,0.931854,0.832857,0.951747,0.963732,0.0,0.0,...,0.960882,0.984122,0.978782,0.943735,0.965409,-1.000000,0.966829,0.984020,0.984299,0.940660
99992,0.785205,0.951885,0.953379,0.968734,0.951266,0.861105,0.959221,0.980018,0.0,0.0,...,0.971874,0.974357,0.973359,0.970688,0.985955,0.966829,-1.000000,0.986753,0.987075,0.966643
99993,0.815312,0.945711,0.941838,0.961725,0.950049,0.825243,0.968324,0.978763,0.0,0.0,...,0.977038,0.988719,0.985815,0.961766,0.987279,0.984020,0.986753,-1.000000,0.983733,0.972190
99994,0.765065,0.948236,0.947014,0.961719,0.939495,0.887657,0.947163,0.983575,0.0,0.0,...,0.970198,0.977806,0.971213,0.954155,0.975915,0.984299,0.987075,0.983733,-1.000000,0.950364


In [18]:
epilogos_cosine.values[[np.arange(epilogos_cosine.shape[0])]*2] = -1
epilogos_cosine

/tmp/ipykernel_8224/2741860177.py:1: FutureWarning: Using a non-tuple sequence for multidimensional indexing is deprecated; use `arr[tuple(seq)]` instead of `arr[seq]`. In the future this will be interpreted as an array index, `arr[np.array(seq)]`, which will result either in an error or a different result.
  epilogos_cosine.values[[np.arange(epilogos_cosine.shape[0])]*2] = -1


,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
0,-1.000000,0.999371,0.999461,0.999489,0.999727,0.999594,0.999383,0.999383,0.999510,0.999810,...,0.999250,0.999383,0.999383,0.999383,0.999383,0.999383,0.999383,0.999383,0.999383,0.999383
1,0.999371,-1.000000,0.999954,0.999968,0.999914,0.999962,0.999984,0.999984,0.999982,0.999867,...,0.999861,0.999984,0.999984,0.999984,0.999984,0.999984,0.999984,0.999984,0.999984,0.999984
2,0.999461,0.999954,-1.000000,0.999998,0.999882,0.999904,0.999893,0.999893,0.999918,0.999899,...,0.999883,0.999893,0.999893,0.999893,0.999893,0.999893,0.999893,0.999893,0.999893,0.999893
3,0.999489,0.999968,0.999998,-1.000000,0.999909,0.999930,0.999918,0.999918,0.999942,0.999916,...,0.999877,0.999918,0.999918,0.999918,0.999918,0.999918,0.999918,0.999918,0.999918,0.999918
4,0.999727,0.999914,0.999882,0.999909,-1.000000,0.999986,0.999929,0.999929,0.999967,0.999977,...,0.999731,0.999929,0.999929,0.999929,0.999929,0.999929,0.999929,0.999929,0.999929,0.999929
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,0.999383,0.999984,0.999893,0.999918,0.999929,0.999978,1.000000,1.000000,0.999992,0.999854,...,0.999802,1.000000,1.000000,1.000000,1.000000,-1.000000,1.000000,1.000000,1.000000,1.000000
99992,0.999383,0.999984,0.999893,0.999918,0.999929,0.999978,1.000000,1.000000,0.999992,0.999854,...,0.999802,1.000000,1.000000,1.000000,1.000000,1.000000,-1.000000,1.000000,1.000000,1.000000
99993,0.999383,0.999984,0.999893,0.999918,0.999929,0.999978,1.000000,1.000000,0.999992,0.999854,...,0.999802,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,-1.000000,1.000000,1.000000
99994,0.999383,0.999984,0.999893,0.999918,0.999929,0.999978,1.000000,1.000000,0.999992,0.999854,...,0.999802,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,-1.000000,1.000000


In [30]:
motif_distr = motif_cosine.sample(100)
gata1_motif_cosine = motif_cosine.iloc[96548, :]

In [19]:
dhs_distr = dhs_cosine.sample(100)
gata1_dhs_cosine = dhs_cosine.iloc[96548, :]

epilogos_distr = epilogos_cosine.sample(100)
gata1_epilogos_cosine = epilogos_cosine.iloc[96548, :]

In [31]:
motif_distr = pd.concat([motif_distr.T, gata1_motif_cosine], axis=1).T
motif_distr

,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
12940,0.438584,0.740146,0.843278,0.799825,0.718830,0.603533,0.615041,0.496024,0.0,0.557444,...,0.839527,0.834954,0.848225,0.788923,0.639179,0.767146,0.655599,0.814947,0.812608,0.837247
66057,0.523344,0.800481,0.799845,0.850286,0.747772,0.691953,0.715511,0.598526,0.0,0.586361,...,0.834626,0.842890,0.858273,0.839016,0.654293,0.808665,0.688133,0.854560,0.862809,0.875117
79585,0.631289,0.724494,0.714761,0.819666,0.745855,0.746494,0.803044,0.671060,0.0,0.526022,...,0.812487,0.793029,0.826169,0.842951,0.796985,0.831636,0.732839,0.753334,0.816536,0.783339
32091,0.550343,0.755594,0.773369,0.804246,0.733419,0.706437,0.727913,0.612741,0.0,0.628864,...,0.809588,0.812162,0.837672,0.822464,0.647000,0.797333,0.693738,0.854455,0.858139,0.841204
46250,0.794920,0.604873,0.524311,0.708473,0.768783,0.808010,0.810455,0.832065,0.0,0.384762,...,0.709372,0.667312,0.675231,0.789261,0.694253,0.708652,0.726606,0.590021,0.732518,0.695950
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91656,0.525327,0.782804,0.839240,0.842627,0.769704,0.705536,0.698371,0.598304,0.0,0.567069,...,0.861609,0.857177,0.854991,0.841693,0.672461,0.804989,0.699477,0.836619,0.904626,0.863487
30389,0.561068,0.767009,0.820220,0.836235,0.729365,0.703727,0.714583,0.623467,0.0,0.519990,...,0.854803,0.839741,0.820119,0.861464,0.738674,0.823118,0.701641,0.819623,0.858250,0.840819
69182,0.529913,0.764674,0.810120,0.826321,0.743829,0.734292,0.744416,0.582618,0.0,0.605384,...,0.812526,0.857509,0.838812,0.816374,0.646584,0.781372,0.688212,0.816501,0.841115,0.835114
68498,0.496382,0.774580,0.882517,0.848309,0.767547,0.701994,0.690943,0.566013,0.0,0.593711,...,0.861317,0.866359,0.859494,0.853972,0.656303,0.793748,0.678131,0.838705,0.856656,0.862052


In [32]:
motif_distr.rank(pct=True)

,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
12940,0.158416,0.455446,0.881188,0.485149,0.287129,0.178218,0.138614,0.148515,0.50495,0.732673,...,0.762376,0.683168,0.891089,0.336634,0.247525,0.445545,0.257426,0.722772,0.396040,0.653465
66057,0.366337,0.871287,0.564356,0.881188,0.514851,0.415842,0.455446,0.435644,0.50495,0.900990,...,0.693069,0.772277,0.960396,0.663366,0.356436,0.801980,0.495050,0.970297,0.861386,0.980198
79585,0.722772,0.316832,0.306931,0.603960,0.495050,0.722772,0.821782,0.693069,0.50495,0.584158,...,0.455446,0.376238,0.663366,0.712871,1.000000,0.950495,0.841584,0.267327,0.425743,0.277228
32091,0.504950,0.594059,0.435644,0.524752,0.376238,0.504950,0.514851,0.524752,0.50495,1.000000,...,0.445545,0.495050,0.752475,0.574257,0.297030,0.702970,0.603960,0.960396,0.831683,0.712871
46250,0.960396,0.108911,0.099010,0.148515,0.683168,0.900990,0.891089,0.950495,0.50495,0.118812,...,0.148515,0.138614,0.118812,0.346535,0.702970,0.168317,0.792079,0.099010,0.148515,0.128713
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91656,0.396040,0.821782,0.851485,0.801980,0.693069,0.485149,0.366337,0.425743,0.50495,0.821782,...,0.940594,0.900990,0.950495,0.702970,0.514851,0.762376,0.663366,0.881188,1.000000,0.910891
30389,0.544554,0.712871,0.673267,0.732673,0.326733,0.465347,0.445545,0.564356,0.50495,0.554455,...,0.891089,0.742574,0.574257,0.881188,0.881188,0.910891,0.683168,0.801980,0.841584,0.702970
69182,0.415842,0.693069,0.653465,0.683168,0.455446,0.633663,0.574257,0.376238,0.50495,0.960396,...,0.465347,0.910891,0.762376,0.514851,0.287129,0.594059,0.504950,0.742574,0.663366,0.613861
68498,0.326733,0.762376,1.000000,0.851485,0.663366,0.445545,0.326733,0.326733,0.50495,0.920792,...,0.930693,0.970297,0.970297,0.811881,0.386139,0.643564,0.425743,0.920792,0.821782,0.891089


In [33]:
print(motif_distr.loc[96548, :].idxmax())
print(motif_distr.loc[96548, :].max())

44030
0.9369133102835367


In [32]:
dhs_distr

,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
14433,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
79127,0.768845,0.932405,0.922267,0.941807,0.926971,0.792039,0.939489,0.951170,0.0,0.0,...,0.943141,0.942484,0.968889,0.922785,0.950909,0.939893,0.953714,0.960128,0.946620,0.936565
86952,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
80667,0.759501,0.947191,0.936356,0.952281,0.924488,0.922503,0.948160,0.980701,0.0,0.0,...,0.969055,0.958487,0.964859,0.941633,0.967053,0.964502,0.974744,0.962000,0.988175,0.937173
9410,0.718427,0.880267,0.866657,0.882561,0.870476,0.978406,0.890375,0.932915,0.0,0.0,...,0.925292,0.898521,0.902045,0.892848,0.917651,0.908490,0.931118,0.910140,0.949332,0.874230
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28477,0.802614,0.974788,0.974988,0.987217,0.963842,0.824689,0.971709,0.984491,0.0,0.0,...,0.978308,0.978792,0.985473,0.948672,0.987168,0.976951,0.982639,0.988307,0.981827,0.970144
51163,0.756289,0.939779,0.941032,0.959944,0.949315,0.810857,0.950696,0.955330,0.0,0.0,...,0.943541,0.943767,0.974390,0.928797,0.956019,0.944229,0.960379,0.959734,0.955002,0.937627
38251,0.778540,0.972003,0.953122,0.972619,0.945630,0.841317,0.972179,0.995173,0.0,0.0,...,0.979933,0.972932,0.980733,0.949390,0.977394,0.965685,0.979664,0.978008,0.979002,0.968473
56711,0.776765,0.929364,0.933500,0.952103,0.946693,0.917487,0.956024,0.974520,0.0,0.0,...,0.968695,0.962224,0.965316,0.949904,0.975900,0.956934,0.985856,0.969334,0.985011,0.942916


In [20]:
dhs_distr = pd.concat([dhs_distr.T, gata1_dhs_cosine], axis=1).T
dhs_distr


,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
23273,0.797383,0.975992,0.963554,0.974085,0.939132,0.811877,0.961725,0.979042,0.0,0.0,...,0.973204,0.968646,0.975757,0.973183,0.981614,0.963994,0.983306,0.980473,0.972697,0.959571
18237,0.737223,0.905984,0.894993,0.918479,0.907805,0.843435,0.927540,0.948831,0.0,0.0,...,0.933700,0.962684,0.940497,0.925387,0.949948,0.937437,0.972361,0.962680,0.964758,0.938895
19942,0.769607,0.936234,0.927786,0.967053,0.956253,0.798183,0.979550,0.960600,0.0,0.0,...,0.950776,0.961910,0.983053,0.938892,0.962874,0.951913,0.965051,0.965845,0.954548,0.956966
71240,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
30820,0.762594,0.945319,0.940100,0.953602,0.936277,0.872807,0.944850,0.973417,0.0,0.0,...,0.962278,0.968957,0.964064,0.973759,0.974249,0.968921,0.994510,0.980708,0.988700,0.945883
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5562,0.842545,0.961546,0.951853,0.971767,0.941119,0.809816,0.982391,0.979135,0.0,0.0,...,0.983029,0.976739,0.985397,0.942145,0.990040,0.962006,0.975178,0.984632,0.966549,0.976352
92565,0.770836,0.992591,0.972251,0.980455,0.930786,0.811158,0.953787,0.979508,0.0,0.0,...,0.966376,0.957044,0.966231,0.941052,0.969247,0.953239,0.968559,0.964175,0.966672,0.952376
96881,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
29290,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [21]:
epilogos_distr = pd.concat([epilogos_distr.T, gata1_epilogos_cosine], axis=1).T
epilogos_distr

,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
30980,0.300486,0.302446,0.302633,0.302841,0.302788,0.302581,0.302287,0.302287,0.302486,0.303123,...,0.289894,0.302287,0.302287,0.302287,0.302287,0.302287,0.302287,0.302287,0.302287,0.302287
20590,-0.187239,-0.191304,-0.197748,-0.196028,-0.186359,-0.186551,-0.186793,-0.186793,-0.187443,-0.190246,...,-0.202796,-0.186793,-0.186793,-0.186793,-0.186793,-0.186793,-0.186793,-0.186793,-0.186793,-0.186793
85092,0.926344,0.926979,0.927351,0.927384,0.927220,0.927040,0.926770,0.926770,0.926969,0.927545,...,0.921699,0.926770,0.926770,0.926770,0.926770,0.926770,0.926770,0.926770,0.926770,0.926770
66045,0.984540,0.984120,0.984543,0.984542,0.984484,0.984268,0.983943,0.983943,0.984167,0.984818,...,0.981840,0.983943,0.983943,0.983943,0.983943,0.983943,0.983943,0.983943,0.983943,0.983943
7174,0.998606,0.998693,0.998998,0.998939,0.998617,0.998581,0.998495,0.998495,0.998587,0.998805,...,0.999334,0.998495,0.998495,0.998495,0.998495,0.998495,0.998495,0.998495,0.998495,0.998495
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47321,0.999421,0.999129,0.999310,0.999314,0.999352,0.999241,0.999059,0.999059,0.999183,0.999490,...,0.998678,0.999059,0.999059,0.999059,0.999059,0.999059,0.999059,0.999059,0.999059,0.999059
29877,0.999383,0.999984,0.999893,0.999918,0.999929,0.999978,1.000000,1.000000,0.999992,0.999854,...,0.999802,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
4818,0.999383,0.999984,0.999893,0.999918,0.999929,0.999978,1.000000,1.000000,0.999992,0.999854,...,0.999802,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25699,0.999383,0.999984,0.999893,0.999918,0.999929,0.999978,1.000000,1.000000,0.999992,0.999854,...,0.999802,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [22]:
dhs_distr.rank(pct=True)


,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
23273,0.831683,0.950495,0.891089,0.811881,0.504950,0.425743,0.603960,0.613861,0.50495,0.50495,...,0.722772,0.504950,0.445545,0.920792,0.821782,0.475248,0.732673,0.594059,0.584158,0.584158
18237,0.108911,0.108911,0.108911,0.108911,0.128713,0.821782,0.128713,0.148515,0.50495,0.50495,...,0.118812,0.356436,0.118812,0.158416,0.148515,0.138614,0.346535,0.178218,0.287129,0.158416
19942,0.396040,0.188119,0.227723,0.594059,0.861386,0.148515,0.930693,0.207921,0.50495,0.50495,...,0.227723,0.336634,0.782178,0.237624,0.277228,0.237624,0.227723,0.198020,0.198020,0.495050
71240,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554,0.50495,0.50495,...,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554
30820,0.227723,0.356436,0.415842,0.247525,0.425743,0.970297,0.287129,0.495050,0.50495,0.50495,...,0.415842,0.524752,0.217822,0.940594,0.564356,0.673267,1.000000,0.603960,0.970297,0.247525
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5562,0.990099,0.712871,0.712871,0.762376,0.534653,0.366337,0.990099,0.623762,0.50495,0.50495,...,0.960396,0.881188,0.841584,0.306931,0.990099,0.435644,0.475248,0.811881,0.316832,0.900990
92565,0.435644,1.000000,0.980198,0.920792,0.326733,0.415842,0.425743,0.633663,0.50495,0.50495,...,0.524752,0.257426,0.257426,0.277228,0.435644,0.277228,0.277228,0.188119,0.336634,0.346535
96881,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554,0.50495,0.50495,...,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554
29290,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554,0.50495,0.50495,...,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554,0.044554


In [26]:
print(dhs_distr.loc[96548, :].idxmax())
print(dhs_distr.loc[96548, :].max())

85205
0.9977672315857422


In [23]:
epilogos_distr.rank(pct=True)

,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
30980,0.425743,0.425743,0.425743,0.425743,0.425743,0.425743,0.425743,0.425743,0.425743,0.425743,...,0.425743,0.425743,0.425743,0.425743,0.425743,0.425743,0.425743,0.425743,0.425743,0.425743
20590,0.118812,0.099010,0.079208,0.099010,0.118812,0.118812,0.108911,0.108911,0.108911,0.118812,...,0.069307,0.108911,0.108911,0.108911,0.108911,0.108911,0.108911,0.108911,0.108911,0.108911
85092,0.603960,0.603960,0.603960,0.603960,0.603960,0.603960,0.603960,0.603960,0.603960,0.603960,...,0.603960,0.603960,0.603960,0.603960,0.603960,0.603960,0.603960,0.603960,0.603960,0.603960
66045,0.643564,0.643564,0.643564,0.643564,0.643564,0.643564,0.643564,0.643564,0.643564,0.643564,...,0.643564,0.643564,0.643564,0.643564,0.643564,0.643564,0.643564,0.643564,0.643564,0.643564
7174,0.821782,0.831683,0.841584,0.841584,0.821782,0.821782,0.831683,0.831683,0.831683,0.821782,...,0.871287,0.831683,0.831683,0.831683,0.831683,0.831683,0.831683,0.831683,0.831683,0.831683
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47321,0.960396,0.881188,0.871287,0.881188,0.881188,0.881188,0.871287,0.871287,0.881188,0.891089,...,0.831683,0.871287,0.871287,0.871287,0.871287,0.871287,0.871287,0.871287,0.871287,0.871287
29877,0.905941,0.965347,0.965347,0.965347,0.955446,0.965347,0.965347,0.965347,0.965347,0.955446,...,0.955446,0.965347,0.965347,0.965347,0.965347,0.965347,0.965347,0.965347,0.965347,0.965347
4818,0.905941,0.965347,0.965347,0.965347,0.955446,0.965347,0.965347,0.965347,0.965347,0.955446,...,0.955446,0.965347,0.965347,0.965347,0.965347,0.965347,0.965347,0.965347,0.965347,0.965347
25699,0.905941,0.965347,0.965347,0.965347,0.955446,0.965347,0.965347,0.965347,0.965347,0.955446,...,0.955446,0.965347,0.965347,0.965347,0.965347,0.965347,0.965347,0.965347,0.965347,0.965347


In [27]:
print(epilogos_distr.loc[96548, :].idxmax())
print(epilogos_distr.loc[96548, :].max())

66854
0.9941503736945521


In [4]:
motif_cosine = pd.read_pickle('../output/motif_cosine_pct.pkl')
combined_score = 1-motif_cosine
del motif_cosine

dhs_cosine = pd.read_pickle('../output/dhs_cosine_pct.pkl')
combined_score = combined_score * (1-dhs_cosine)
del dhs_cosine

epilogos_cosine = pd.read_pickle('../output/epilogos_cosine_pct.pkl')
combined_score = combined_score * (1-epilogos_cosine)
del epilogos_cosine

combined_score

,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
0,0.999970,0.107544,1.107016e-01,1.075946e-01,0.042513,0.049774,0.021829,0.000008,0.029715,0.048797,...,0.132680,0.103433,0.103606,0.103160,0.031073,0.103078,0.006181,0.103795,0.103415,0.103406
1,0.058886,0.999970,8.077627e-05,2.252527e-04,0.069224,0.057339,0.038144,0.041564,0.023565,0.003155,...,0.007045,0.060101,0.063045,0.063291,0.046919,0.050928,0.047123,0.048035,0.061673,0.059401
2,0.016857,0.003515,9.999700e-01,4.067684e-07,0.010213,0.069631,0.074343,0.072913,0.024720,0.000900,...,0.002738,0.033391,0.058437,0.051377,0.068349,0.064753,0.071811,0.043003,0.054805,0.033862
3,0.007958,0.001341,2.150495e-08,9.999700e-01,0.001285,0.023010,0.026287,0.037150,0.024390,0.001488,...,0.001803,0.007233,0.033081,0.005947,0.029801,0.025802,0.040284,0.046543,0.025255,0.027181
4,0.000618,0.067661,1.302746e-02,2.796487e-02,0.999970,0.000022,0.033118,0.004150,0.023962,0.000416,...,0.080316,0.067294,0.065666,0.067787,0.009656,0.069452,0.000605,0.065024,0.067461,0.065593
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,0.010675,0.014505,2.757886e-02,2.308682e-02,0.026152,0.007142,0.013933,0.013564,0.011691,0.018130,...,0.024021,0.000541,0.001323,0.013797,0.000277,0.999970,0.005707,0.006099,0.001335,0.017466
99992,0.001730,0.018973,1.167705e-02,1.637052e-02,0.000066,0.000883,0.011085,0.000336,0.011691,0.021658,...,0.017200,0.008309,0.021866,0.002853,0.000035,0.013200,0.999970,0.003716,0.001296,0.007067
99993,0.005863,0.006058,1.293760e-02,1.963450e-02,0.009219,0.010215,0.006705,0.009823,0.011691,0.001630,...,0.005078,0.000062,0.000488,0.004188,0.001927,0.001127,0.002947,0.999970,0.001465,0.001426
99994,0.021632,0.008381,9.298995e-03,5.374610e-03,0.009059,0.000234,0.010898,0.002914,0.011691,0.008860,...,0.001460,0.000525,0.003306,0.000344,0.008198,0.000165,0.001113,0.000977,0.999970,0.000389


In [5]:
combined_score.to_pickle('../output/combined_score_cosine_25kb_pct.pkl')

In [3]:
epilogos_cosine.values[[np.arange(epilogos_cosine.shape[0])]*2] = -1
epilogos_cosine

/tmp/ipykernel_19347/2741860177.py:1: FutureWarning: Using a non-tuple sequence for multidimensional indexing is deprecated; use `arr[tuple(seq)]` instead of `arr[seq]`. In the future this will be interpreted as an array index, `arr[np.array(seq)]`, which will result either in an error or a different result.
  epilogos_cosine.values[[np.arange(epilogos_cosine.shape[0])]*2] = -1


,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
0,-1.000000,0.999371,0.999461,0.999489,0.999727,0.999594,0.999383,0.999383,0.999510,0.999810,...,0.999250,0.999383,0.999383,0.999383,0.999383,0.999383,0.999383,0.999383,0.999383,0.999383
1,0.999371,-1.000000,0.999954,0.999968,0.999914,0.999962,0.999984,0.999984,0.999982,0.999867,...,0.999861,0.999984,0.999984,0.999984,0.999984,0.999984,0.999984,0.999984,0.999984,0.999984
2,0.999461,0.999954,-1.000000,0.999998,0.999882,0.999904,0.999893,0.999893,0.999918,0.999899,...,0.999883,0.999893,0.999893,0.999893,0.999893,0.999893,0.999893,0.999893,0.999893,0.999893
3,0.999489,0.999968,0.999998,-1.000000,0.999909,0.999930,0.999918,0.999918,0.999942,0.999916,...,0.999877,0.999918,0.999918,0.999918,0.999918,0.999918,0.999918,0.999918,0.999918,0.999918
4,0.999727,0.999914,0.999882,0.999909,-1.000000,0.999986,0.999929,0.999929,0.999967,0.999977,...,0.999731,0.999929,0.999929,0.999929,0.999929,0.999929,0.999929,0.999929,0.999929,0.999929
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,0.999383,0.999984,0.999893,0.999918,0.999929,0.999978,1.000000,1.000000,0.999992,0.999854,...,0.999802,1.000000,1.000000,1.000000,1.000000,-1.000000,1.000000,1.000000,1.000000,1.000000
99992,0.999383,0.999984,0.999893,0.999918,0.999929,0.999978,1.000000,1.000000,0.999992,0.999854,...,0.999802,1.000000,1.000000,1.000000,1.000000,1.000000,-1.000000,1.000000,1.000000,1.000000
99993,0.999383,0.999984,0.999893,0.999918,0.999929,0.999978,1.000000,1.000000,0.999992,0.999854,...,0.999802,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,-1.000000,1.000000,1.000000
99994,0.999383,0.999984,0.999893,0.999918,0.999929,0.999978,1.000000,1.000000,0.999992,0.999854,...,0.999802,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,-1.000000,1.000000


In [5]:
epilogos_cosine = epilogos_cosine.rank(pct=True)
epilogos_cosine

,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
0,0.000010,0.872515,0.869465,0.872735,0.890636,0.885185,0.877385,0.877385,0.881135,0.893516,...,0.841434,0.877385,0.877385,0.877385,0.877385,0.877385,0.877385,0.877385,0.877385,0.877385
1,0.877935,0.000010,0.997210,0.998100,0.901866,0.904316,0.906556,0.906556,0.905736,0.990510,...,0.988720,0.906556,0.906556,0.906556,0.906556,0.906556,0.906556,0.906556,0.906556,0.906556
2,0.977349,0.903516,0.000010,0.999940,0.899296,0.900186,0.900936,0.900936,0.901116,0.993500,...,0.991540,0.900936,0.900936,0.900936,0.900936,0.900936,0.900936,0.900936,0.900936,0.900936
3,0.979579,0.904806,0.999970,0.000010,0.901396,0.901886,0.902086,0.902086,0.902436,0.994880,...,0.990660,0.902086,0.902086,0.902086,0.902086,0.902086,0.902086,0.902086,0.902086,0.902086
4,0.992380,0.900806,0.897896,0.899716,0.000010,0.998030,0.902596,0.902596,0.904146,0.998830,...,0.883585,0.902596,0.902596,0.902596,0.902596,0.902596,0.902596,0.902596,0.902596,0.902596
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,0.924082,0.953013,0.944643,0.946693,0.949793,0.952763,0.955748,0.955748,0.953233,0.943223,...,0.936632,0.955748,0.955748,0.955748,0.955748,0.000010,0.955748,0.955748,0.955748,0.955748
99992,0.924082,0.953013,0.944643,0.946693,0.949793,0.952763,0.955748,0.955748,0.953233,0.943223,...,0.936632,0.955748,0.955748,0.955748,0.955748,0.955748,0.000010,0.955748,0.955748,0.955748
99993,0.924082,0.953013,0.944643,0.946693,0.949793,0.952763,0.955748,0.955748,0.953233,0.943223,...,0.936632,0.955748,0.955748,0.955748,0.955748,0.955748,0.955748,0.000010,0.955748,0.955748
99994,0.924082,0.953013,0.944643,0.946693,0.949793,0.952763,0.955748,0.955748,0.953233,0.943223,...,0.936632,0.955748,0.955748,0.955748,0.955748,0.955748,0.955748,0.955748,0.000010,0.955748


In [6]:
epilogos_cosine.to_pickle('../output/epilogos_cosine_pct.pkl')

In [2]:
# del epilogos_cosine
dhs_cosine = pd.read_pickle('../data/dhs_cosine_25kb.pkl')
dhs_cosine.values[[np.arange(dhs_cosine.shape[0])]*2] = -1
dhs_cosine = dhs_cosine.rank(pct=True)
dhs_cosine.to_pickle('../output/dhs_cosine_pct.pkl')
combined_score = 1-dhs_cosine
del dhs_cosine

/tmp/ipykernel_35189/110383171.py:3: FutureWarning: Using a non-tuple sequence for multidimensional indexing is deprecated; use `arr[tuple(seq)]` instead of `arr[seq]`. In the future this will be interpreted as an array index, `arr[np.array(seq)]`, which will result either in an error or a different result.
  dhs_cosine.values[[np.arange(dhs_cosine.shape[0])]*2] = -1


In [3]:
motif_cosine = pd.read_pickle('../data/motif_cosine_25kb.pkl')
motif_cosine.values[[np.arange(motif_cosine.shape[0])]*2] = -1
motif_cosine = motif_cosine.rank(pct=True)
motif_cosine.to_pickle('../output/motif_cosine_pct.pkl')
combined_score = combined_score * (1-motif_cosine)

/tmp/ipykernel_35189/2341962081.py:2: FutureWarning: Using a non-tuple sequence for multidimensional indexing is deprecated; use `arr[tuple(seq)]` instead of `arr[seq]`. In the future this will be interpreted as an array index, `arr[np.array(seq)]`, which will result either in an error or a different result.
  motif_cosine.values[[np.arange(motif_cosine.shape[0])]*2] = -1


In [6]:
epilogos_cosine=pd.read_pickle('../output/epilogos_cosine_pct.pkl')
combined_score = combined_score * (1-epilogos_cosine)
del epilogos_cosine

In [8]:
combined_score = 1-combined_score

In [11]:
combined_score.to_pickle('../output/combined_score.pkl')

In [12]:
combined_score

,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
0,0.000030,0.892456,0.889298,0.892405,0.957487,0.950226,0.978171,0.999992,0.970285,0.951203,...,0.867320,0.896567,0.896394,0.896840,0.968927,0.896922,0.993819,0.896205,0.896585,0.896594
1,0.941114,0.000030,0.999919,0.999775,0.930776,0.942661,0.961856,0.958436,0.976435,0.996845,...,0.992955,0.939899,0.936955,0.936709,0.953081,0.949072,0.952877,0.951965,0.938327,0.940599
2,0.983143,0.996485,0.000030,1.000000,0.989787,0.930369,0.925657,0.927087,0.975280,0.999100,...,0.997262,0.966609,0.941563,0.948623,0.931651,0.935247,0.928189,0.956997,0.945195,0.966138
3,0.992042,0.998659,1.000000,0.000030,0.998715,0.976990,0.973713,0.962850,0.975610,0.998512,...,0.998197,0.992767,0.966919,0.994053,0.970199,0.974198,0.959716,0.953457,0.974745,0.972819
4,0.999382,0.932339,0.986973,0.972035,0.000030,0.999978,0.966882,0.995850,0.976038,0.999584,...,0.919684,0.932706,0.934334,0.932213,0.990344,0.930548,0.999395,0.934976,0.932539,0.934407
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,0.989325,0.985495,0.972421,0.976913,0.973848,0.992858,0.986067,0.986436,0.988309,0.981870,...,0.975979,0.999459,0.998677,0.986203,0.999723,0.000030,0.994293,0.993901,0.998665,0.982534
99992,0.998270,0.981027,0.988323,0.983629,0.999934,0.999117,0.988915,0.999664,0.988309,0.978342,...,0.982800,0.991691,0.978134,0.997147,0.999965,0.986800,0.000030,0.996284,0.998704,0.992933
99993,0.994137,0.993942,0.987062,0.980365,0.990781,0.989785,0.993295,0.990177,0.988309,0.998370,...,0.994922,0.999938,0.999512,0.995812,0.998073,0.998873,0.997053,0.000030,0.998535,0.998574
99994,0.978368,0.991619,0.990701,0.994625,0.990941,0.999766,0.989102,0.997086,0.988309,0.991140,...,0.998540,0.999475,0.996694,0.999656,0.991802,0.999835,0.998887,0.999023,0.000030,0.999611


In [13]:
print(combined_score.loc[96548].idxmax())
print(combined_score.loc[96548].max())

62105
0.9999999472885749


In [9]:
STRING_euclidean_df_pct = STRING_euclidean_df.rank(pct=True)
STRING_euclidean_df_pct

MemoryError: Unable to allocate 74.5 GiB for an array with shape (99996, 99996) and data type float64

In [10]:
combined_score.to_pickle('../output/combined_score.pkl')

In [11]:
combined_score

,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.051377,NaN,NaN,NaN,0.043003,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99992,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99993,NaN,NaN,0.012938,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.004188,NaN,NaN,NaN,NaN,NaN,NaN
99994,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
STRING_euclidean_df = pd.read_pickle('../output/STRING_cosine_25kb.pkl')

In [4]:
1-STRING_euclidean_df

,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,-431.482370,NaN,NaN,NaN,-1165.242256,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99992,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99993,NaN,NaN,-1165.242256,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,-837.379389,NaN,NaN,NaN,1.000000,NaN,NaN
99994,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
STRING_euclidean_df_pct = STRING_euclidean_df.rank(pct=True)
STRING_euclidean_df_pct

,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,0.000051,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.120530,NaN,NaN,NaN,0.871268,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99992,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99993,NaN,NaN,0.809373,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.464391,NaN,NaN,NaN,0.000051,NaN,NaN
99994,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
STRING_euclidean_df_pct = pd.read_pickle('../output/STRING_euclidean_25kb_pct.pkl')
# combined_score = pd.read_pickle('../output/combined_score_cosine_25kb_pct.pkl')

In [7]:
# Then subtract from 1 after you get percentages
STRING_1_minus_euclidean_df_pct = (1-STRING_euclidean_df_pct).fillna(0)
del STRING_euclidean_df_pct
combined_score = combined_score * STRING_1_minus_euclidean_df_pct

In [8]:
combined_score

,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
1,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
2,0.0,0.0,0.999919,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.045185,0.0,0.0,0.0,0.005536,0.0,0.0
3,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
4,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
99992,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
99993,0.0,0.0,0.002466,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.002243,0.0,0.0,0.0,0.999919,0.0,0.0
99994,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0


In [6]:
print(combined_score_STRING.iloc[96548, :].idxmax())
print(combined_score_STRING.iloc[96548, :].max())

96548
0.9999190581932456


In [9]:
STRING_1_minus_euclidean_df_pct
print(STRING_1_minus_euclidean_df_pct.iloc[96548, :].idxmax())
print(STRING_1_minus_euclidean_df_pct.iloc[96548, :].max())

96548
0.9999490575649516


In [2]:
STRING_euclidean_df = pd.read_pickle('../output/STRING_cosine_25kb.pkl')
STRING_euclidean_df.values[[np.arange(STRING_euclidean_df.shape[0])]*2] = -1
STRING_euclidean_df

/tmp/ipykernel_72699/2108377125.py:2: FutureWarning: Using a non-tuple sequence for multidimensional indexing is deprecated; use `arr[tuple(seq)]` instead of `arr[seq]`. In the future this will be interpreted as an array index, `arr[np.array(seq)]`, which will result either in an error or a different result.
  STRING_euclidean_df.values[[np.arange(STRING_euclidean_df.shape[0])]*2] = -1


,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,432.482370,NaN,NaN,NaN,1166.242256,NaN,NaN
3,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,NaN
99992,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,NaN,NaN
99993,NaN,NaN,1166.242256,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,838.379389,NaN,NaN,NaN,-1.000000,NaN,NaN
99994,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN


In [7]:
# STRING_euclidean_df_pct = STRING_euclidean_df.rank(pct=True)
# print(STRING_euclidean_df_pct)
STRING_euclidean_df_pct.values[[np.arange(STRING_euclidean_df.shape[0])]*2] = 0
STRING_euclidean_df_pct.fillna(0)

/tmp/ipykernel_72699/3102769461.py:3: FutureWarning: Using a non-tuple sequence for multidimensional indexing is deprecated; use `arr[tuple(seq)]` instead of `arr[seq]`. In the future this will be interpreted as an array index, `arr[np.array(seq)]`, which will result either in an error or a different result.
  STRING_euclidean_df_pct.values[[np.arange(STRING_euclidean_df.shape[0])]*2] = 0


,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
1,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
2,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.120530,0.0,0.0,0.0,0.871268,0.0,0.0
3,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
4,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
99992,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
99993,0.0,0.0,0.809373,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.464391,0.0,0.0,0.0,0.000000,0.0,0.0
99994,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0


In [2]:
# STRING_euclidean_df_pct = STRING_euclidean_df_pct.fillna(0)
STRING_euclidean_df_pct = pd.read_pickle('../output/STRING_euclidean_25kb_pct.pkl')

In [10]:
# STRING_1_minus_euclidean_df_pct = (1-STRING_euclidean_df_pct).fillna(0)
# STRING_1_minus_euclidean_df_pct = STRING_euclidean_df_pct.fillna(0)
# print(STRING_1_minus_euclidean_df_pct)
# del STRING_euclidean_df
# combined_score = pd.read_pickle('../output/combined_score_cosine_25kb_pct.pkl')
combined_score_STRING = 1-((1-combined_score) * (1-STRING_euclidean_df_pct))

MemoryError: Unable to allocate 74.5 GiB for an array with shape (99996, 99996) and data type float64

In [6]:
combined_score_STRING

,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
0,0.999970,0.107544,1.107016e-01,1.075946e-01,0.042513,0.049774,0.021829,0.000008,0.029715,0.048797,...,0.132680,0.103433,0.103606,0.103160,0.031073,0.103078,0.006181,0.103795,0.103415,0.103406
1,0.058886,0.999970,8.077627e-05,2.252527e-04,0.069224,0.057339,0.038144,0.041564,0.023565,0.003155,...,0.007045,0.060101,0.063045,0.063291,0.046919,0.050928,0.047123,0.048035,0.061673,0.059401
2,0.016857,0.003515,9.999700e-01,4.067684e-07,0.010213,0.069631,0.074343,0.072913,0.024720,0.000900,...,0.002738,0.033391,0.058437,0.045185,0.068349,0.064753,0.071811,0.005536,0.054805,0.033862
3,0.007958,0.001341,2.150495e-08,9.999700e-01,0.001285,0.023010,0.026287,0.037150,0.024390,0.001488,...,0.001803,0.007233,0.033081,0.005947,0.029801,0.025802,0.040284,0.046543,0.025255,0.027181
4,0.000618,0.067661,1.302746e-02,2.796487e-02,0.999970,0.000022,0.033118,0.004150,0.023962,0.000416,...,0.080316,0.067294,0.065666,0.067787,0.009656,0.069452,0.000605,0.065024,0.067461,0.065593
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,0.010675,0.014505,2.757886e-02,2.308682e-02,0.026152,0.007142,0.013933,0.013564,0.011691,0.018130,...,0.024021,0.000541,0.001323,0.013797,0.000277,0.999970,0.005707,0.006099,0.001335,0.017466
99992,0.001730,0.018973,1.167705e-02,1.637052e-02,0.000066,0.000883,0.011085,0.000336,0.011691,0.021658,...,0.017200,0.008309,0.021866,0.002853,0.000035,0.013200,0.999970,0.003716,0.001296,0.007067
99993,0.005863,0.006058,2.466251e-03,1.963450e-02,0.009219,0.010215,0.006705,0.009823,0.011691,0.001630,...,0.005078,0.000062,0.000488,0.002243,0.001927,0.001127,0.002947,0.999970,0.001465,0.001426
99994,0.021632,0.008381,9.298995e-03,5.374610e-03,0.009059,0.000234,0.010898,0.002914,0.011691,0.008860,...,0.001460,0.000525,0.003306,0.000344,0.008198,0.000165,0.001113,0.000977,0.999970,0.000389


In [7]:
print(combined_score_STRING.iloc[96548, :].idxmax())
print(combined_score_STRING.iloc[96548, :].max())

57993
0.999999979925569


In [9]:
combined_score

,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
0,0.999970,0.107544,1.107016e-01,1.075946e-01,0.042513,0.049774,0.021829,0.000008,0.029715,0.048797,...,0.132680,0.103433,0.103606,0.103160,0.031073,0.103078,0.006181,0.103795,0.103415,0.103406
1,0.058886,0.999970,8.077627e-05,2.252527e-04,0.069224,0.057339,0.038144,0.041564,0.023565,0.003155,...,0.007045,0.060101,0.063045,0.063291,0.046919,0.050928,0.047123,0.048035,0.061673,0.059401
2,0.016857,0.003515,9.999700e-01,4.067684e-07,0.010213,0.069631,0.074343,0.072913,0.024720,0.000900,...,0.002738,0.033391,0.058437,0.051377,0.068349,0.064753,0.071811,0.043003,0.054805,0.033862
3,0.007958,0.001341,2.150495e-08,9.999700e-01,0.001285,0.023010,0.026287,0.037150,0.024390,0.001488,...,0.001803,0.007233,0.033081,0.005947,0.029801,0.025802,0.040284,0.046543,0.025255,0.027181
4,0.000618,0.067661,1.302746e-02,2.796487e-02,0.999970,0.000022,0.033118,0.004150,0.023962,0.000416,...,0.080316,0.067294,0.065666,0.067787,0.009656,0.069452,0.000605,0.065024,0.067461,0.065593
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,0.010675,0.014505,2.757886e-02,2.308682e-02,0.026152,0.007142,0.013933,0.013564,0.011691,0.018130,...,0.024021,0.000541,0.001323,0.013797,0.000277,0.999970,0.005707,0.006099,0.001335,0.017466
99992,0.001730,0.018973,1.167705e-02,1.637052e-02,0.000066,0.000883,0.011085,0.000336,0.011691,0.021658,...,0.017200,0.008309,0.021866,0.002853,0.000035,0.013200,0.999970,0.003716,0.001296,0.007067
99993,0.005863,0.006058,1.293760e-02,1.963450e-02,0.009219,0.010215,0.006705,0.009823,0.011691,0.001630,...,0.005078,0.000062,0.000488,0.004188,0.001927,0.001127,0.002947,0.999970,0.001465,0.001426
99994,0.021632,0.008381,9.298995e-03,5.374610e-03,0.009059,0.000234,0.010898,0.002914,0.011691,0.008860,...,0.001460,0.000525,0.003306,0.000344,0.008198,0.000165,0.001113,0.000977,0.999970,0.000389


In [8]:
STRING_euclidean_df_pct

,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
1,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
2,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.120530,0.0,0.0,0.0,0.871268,0.0,0.0
3,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
4,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
99992,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
99993,0.0,0.0,0.809373,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.464391,0.0,0.0,0.0,0.000000,0.0,0.0
99994,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
